# Лабораторная работа №1

Исследование, сериализация, оптимизация типов и очистка демографических данных. В папке raw сейчас демонстрационный синтетический срез; замените его исходными данными перед окончательной сдачей.

In [ ]:
import sys, pandas as pd, numpy as np
print(sys.version)
print(pd.__version__)

In [ ]:
from pathlib import Path
path=Path('../data/raw/sample_demographics.csv')
df=pd.read_csv(path)
print(df.shape)
df.head()

In [ ]:
df.info(memory_usage='deep')
df.isna().mean().sort_values(ascending=False)

## Сериализация
Замеры форматов выполняются отдельным скриптом `src/benchmark.py` с медианой трёх измерений.

In [ ]:
df.to_parquet('../data/interim/source.parquet', index=False)
df.to_feather('../data/interim/source.feather')
df.to_pickle('../data/interim/source.pkl')

## Очистка
Очистка реализована в модуле и повторно используется конвейером CLI.

In [ ]:
import sys
sys.path.insert(0,'..')
from src.cleaning import clean_frame
cleaned, log=clean_frame(df)
print('до:',len(df),'после:',len(cleaned))
cleaned.head()

In [ ]:
cleaned.to_parquet('../data/processed/clean.parquet', index=False)
pd.DataFrame(log)

## Группировка


In [ ]:
num=cleaned.select_dtypes('number').columns
if len(num):
    display(cleaned.groupby('country')[num[0]].median().sort_values(ascending=False).head(10))

## СУБД
Конвейер загружает результат в SQLite (`data/processed/lab1.sqlite`). PostgreSQL загрузка предусмотрена отдельно через `DATABASE_URL` и модуль `src.db`.

In [ ]:
from src.db import sqlite_load
con=sqlite_load(cleaned)
con.close()
print('SQLite database created')